In [1]:
%reset -f

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os

torch.set_default_dtype(torch.float64)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## 1. DEFINE NEURAL NETWORK ARCHITECTURE

In [3]:
# Define the same Net architecture as NN_defs.ipynb
class Net(nn.Module):
    def __init__(self, d_in=4, nodes=64, layers=4, d_out=1):
        super(Net, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(d_in, nodes))  # First layer
        for _ in range(1, layers):  # Remaining hidden layers
            self.layers.append(nn.Linear(nodes, nodes))
        self.out = nn.Linear(nodes, d_out)
    
    def forward(self, x):
        for layer in self.layers:
            x = torch.sigmoid(layer(x))
        return self.out(x)

print('✓ Net architecture defined')

✓ Net architecture defined


## 2. LOAD MODEL CHECKPOINT

In [4]:
# Model path (modify if needed)
model_file = './model/2layer_nz640_tf20_random.pt'

# Model hyperparameters (must match training)
d_in = 4
d_out = 1
layers = 4
nodes = 64

print("="*80)
print("LOADING TRAINED MODEL")
print("="*80)

# Create model
model = Net(d_in=d_in, nodes=nodes, layers=layers, d_out=d_out)
model = model.to(device)

# Load checkpoint
try:
    checkpoint = torch.load(model_file, weights_only=False)
    
    # Extract model_dict if checkpoint is a dict with multiple keys
    if isinstance(checkpoint, dict) and 'model_dict' in checkpoint:
        model_dict = checkpoint['model_dict']
    else:
        model_dict = checkpoint
    
    model.load_state_dict(model_dict)
    model.eval()  # Set to evaluation mode
    
    print(f"✓ Model loaded from: {model_file}")
    print(f"  Architecture: {layers} hidden layers, {nodes} nodes each")
    print(f"  Input dim: {d_in}, Output dim: {d_out}")
    print(f"  Model on device: {next(model.parameters()).device}")
    
except FileNotFoundError:
    print(f"✗ Error: Model file not found at {model_file}")
    print(f"  Please check the file path and make sure training is complete.")
    raise

LOADING TRAINED MODEL
✓ Model loaded from: ./model/2layer_nz640_tf20_random.pt
  Architecture: 4 hidden layers, 64 nodes each
  Input dim: 4, Output dim: 1
  Model on device: cuda:0


## 3. LOAD GENERATED TRAINING DATA

In [5]:
# Data file path (modify if using different version)
# Options:
#   - Uniform grid: './data/2layer_k1{k1}_k2{k2}_nz{nz}.pt'
#   - Random samples: './data/2layer_k1{k1}_k2{k2}_nz{nz}_random_N{N}.pt'

dat_file = './data/2layer_k11.0_k22.0_nz640_random_N8100.pt'

print("\n" + "="*80)
print("LOADING TRAINING DATA")
print("="*80)

try:
    # Load data
    trn_dat = torch.load(dat_file, weights_only=True)
    
    # Extract and prepare data
    stau_test = trn_dat['stau'].to(device, dtype=torch.float64)
    W_test = trn_dat['W'].to(device, dtype=torch.float64)
    kratio_test = trn_dat['kratio'].to(device, dtype=torch.float64)
    z_disv_test = trn_dat['zdis'].to(device, dtype=torch.float64)
    
    # Stack to 4D input
    train = torch.stack([stau_test[:,0], stau_test[:,1], kratio_test, z_disv_test], dim=-1)
    xtrain = train.to(torch.float64)
    ytrain = W_test.to(torch.float64)
    
    N_data = xtrain.shape[0]
    
    print(f"✓ Data loaded from: {dat_file}")
    print(f"  Total samples: {N_data:,}")
    print(f"  Input shape: {xtrain.shape}  (samples, 4)")
    print(f"  Output shape: {ytrain.shape}  (samples, 1)")
    print(f"  Data on device: {xtrain.device}")
    
except FileNotFoundError:
    print(f"✗ Error: Data file not found at {dat_file}")
    print(f"  Please check the file path and make sure data generation is complete.")
    raise


LOADING TRAINING DATA
✓ Data loaded from: ./data/2layer_k11.0_k22.0_nz640_random_N8100.pt
  Total samples: 8,100
  Input shape: torch.Size([8100, 4])  (samples, 4)
  Output shape: torch.Size([8100, 1])  (samples, 1)
  Data on device: cuda:0


## 4. MODEL PREDICTION & ACCURACY EVALUATION

In [6]:
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

# Make predictions
with torch.no_grad():
    y_pred = model(xtrain)  # Shape: (N_data, 1)

# Compute errors
error = y_pred - ytrain  # Point-wise error
abs_error = torch.abs(error)

# Metrics
mse = torch.mean(error ** 2).item()
rmse = torch.sqrt(mse).item()
mae = torch.mean(abs_error).item()
max_abs_error = torch.max(abs_error).item()
min_abs_error = torch.min(abs_error).item()

# Relative errors
rel_error = abs_error / (torch.abs(ytrain) + 1e-10)  # Avoid division by zero
rel_error_mean = torch.mean(rel_error).item()
rel_error_max = torch.max(rel_error).item()

# R² score
ss_res = torch.sum(error ** 2).item()
ss_tot = torch.sum((ytrain - ytrain.mean()) ** 2).item()
r2_score = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

print(f"\n✓ Predictions complete")
print(f"\n  Absolute Error Metrics:")
print(f"    - MSE (Mean Squared Error):     {mse:.6e}")
print(f"    - RMSE (Root MSE):              {rmse:.6e}")
print(f"    - MAE (Mean Absolute Error):    {mae:.6e}")
print(f"    - Max Absolute Error:           {max_abs_error:.6e}")
print(f"    - Min Absolute Error:           {min_abs_error:.6e}")

print(f"\n  Relative Error Metrics:")
print(f"    - Mean Relative Error:          {rel_error_mean:.6e}")
print(f"    - Max Relative Error:           {rel_error_max:.6e}")

print(f"\n  Goodness of Fit:")
print(f"    - R² Score:                     {r2_score:.6f}")

# Output value statistics
print(f"\n  Output Statistics:")
print(f"    - True (ytrain):")
print(f"      min={ytrain.min():.6e}, max={ytrain.max():.6e}")
print(f"      mean={ytrain.mean():.6e}, std={ytrain.std():.6e}")
print(f"    - Predicted (y_pred):")
print(f"      min={y_pred.min():.6e}, max={y_pred.max():.6e}")
print(f"      mean={y_pred.mean():.6e}, std={y_pred.std():.6e}")


MODEL EVALUATION


TypeError: sqrt(): argument 'input' (position 1) must be Tensor, not float

## 5. ERROR ANALYSIS & PERCENTILES

In [ ]:
print("\n" + "="*80)
print("ERROR PERCENTILES")
print("="*80)

# Compute percentiles
percentiles = [50, 68, 90, 95, 99]
abs_error_cpu = abs_error.cpu().numpy().flatten()

print(f"\n  Absolute Error Percentiles:")
for p in percentiles:
    val = np.percentile(abs_error_cpu, p)
    print(f"    - {p:2d}th percentile: {val:.6e}")

# Relative error percentiles
rel_error_cpu = rel_error.cpu().numpy().flatten()

print(f"\n  Relative Error Percentiles:")
for p in percentiles:
    val = np.percentile(rel_error_cpu, p)
    print(f"    - {p:2d}th percentile: {val:.6e}")

## 6. VISUALIZATIONS: PREDICTIONS vs ACTUAL

In [ ]:
# Move to CPU for visualization
ytrain_cpu = ytrain.cpu().numpy().flatten()
y_pred_cpu = y_pred.cpu().numpy().flatten()
error_cpu = error.cpu().numpy().flatten()
abs_error_cpu = abs_error.cpu().numpy().flatten()
xtrain_cpu = xtrain.cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# [1,1] Predicted vs True (scatter)
ax = axes[0, 0]
ax.scatter(ytrain_cpu, y_pred_cpu, alpha=0.4, s=10)
lims = [min(ytrain_cpu.min(), y_pred_cpu.min()), max(ytrain_cpu.max(), y_pred_cpu.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('True Output', fontsize=11, fontweight='bold')
ax.set_ylabel('Predicted Output', fontsize=11, fontweight='bold')
ax.set_title('Predicted vs True Output', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# [1,2] Residuals (error) distribution
ax = axes[0, 1]
ax.hist(error_cpu, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
ax.set_xlabel('Prediction Error (Predicted - True)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Error Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# [1,3] Absolute error distribution (log scale)
ax = axes[0, 2]
ax.hist(abs_error_cpu, bins=50, alpha=0.7, edgecolor='black')
ax.set_xlabel('Absolute Error |Predicted - True|', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_yscale('log')
ax.set_title('Absolute Error Distribution (log scale)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# [2,1] Error vs True output
ax = axes[1, 0]
scatter = ax.scatter(ytrain_cpu, error_cpu, c=abs_error_cpu, cmap='viridis', 
                     alpha=0.5, s=10)
ax.axhline(0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('True Output', fontsize=11, fontweight='bold')
ax.set_ylabel('Prediction Error', fontsize=11, fontweight='bold')
ax.set_title('Error vs True Output', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('|Error|', fontsize=10)
ax.grid(True, alpha=0.3)

# [2,2] Relative error distribution
ax = axes[1, 1]
rel_error_cpu_clipped = np.clip(rel_error_cpu, 0, np.percentile(rel_error_cpu, 95))
ax.hist(rel_error_cpu_clipped, bins=50, alpha=0.7, edgecolor='black')
ax.set_xlabel('Relative Error |Error|/|True| (clipped at 95%ile)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Relative Error Distribution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# [2,3] Sample-by-sample true vs predicted
ax = axes[1, 2]
sample_idx = np.arange(min(1000, len(ytrain_cpu)))
ax.plot(sample_idx, ytrain_cpu[sample_idx], 'b-', linewidth=1, alpha=0.7, label='True')
ax.plot(sample_idx, y_pred_cpu[sample_idx], 'r--', linewidth=1, alpha=0.7, label='Predicted')
ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax.set_ylabel('Output Value', fontsize=11, fontweight='bold')
ax.set_title(f'True vs Predicted (first 1000 samples)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualization saved: model_evaluation.png')

## 7. INPUT SPACE ANALYSIS: WHERE ARE LARGEST ERRORS?

In [ ]:
print("\n" + "="*80)
print("ERROR LOCALIZATION IN INPUT SPACE")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# [1,1] (s, τ) colored by absolute error
ax = axes[0, 0]
scatter = ax.scatter(xtrain_cpu[:, 0], xtrain_cpu[:, 1], c=abs_error_cpu, 
                     cmap='hot', s=20, alpha=0.6)
ax.set_xlabel('s', fontsize=11, fontweight='bold')
ax.set_ylabel('τ', fontsize=11, fontweight='bold')
ax.set_title('Absolute Error in (s, τ) space', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('|Error|', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# [1,2] s vs absolute error
ax = axes[0, 1]
ax.scatter(xtrain_cpu[:, 0], abs_error_cpu, alpha=0.4, s=10)
ax.set_xlabel('s (input)', fontsize=11, fontweight='bold')
ax.set_ylabel('Absolute Error', fontsize=11, fontweight='bold')
ax.set_title('Error vs s component', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')

# [2,1] τ vs absolute error
ax = axes[1, 0]
ax.scatter(xtrain_cpu[:, 1], abs_error_cpu, alpha=0.4, s=10)
ax.set_xlabel('τ (input)', fontsize=11, fontweight='bold')
ax.set_ylabel('Absolute Error', fontsize=11, fontweight='bold')
ax.set_title('Error vs τ component', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')

# [2,2] Top 10% errors
ax = axes[1, 1]
top_10_threshold = np.percentile(abs_error_cpu, 90)
top_10_mask = abs_error_cpu >= top_10_threshold
ax.scatter(xtrain_cpu[~top_10_mask, 0], xtrain_cpu[~top_10_mask, 1], 
          alpha=0.2, s=10, label='Bottom 90%', color='blue')
ax.scatter(xtrain_cpu[top_10_mask, 0], xtrain_cpu[top_10_mask, 1], 
          alpha=0.8, s=20, label='Top 10% errors', color='red')
ax.set_xlabel('s', fontsize=11, fontweight='bold')
ax.set_ylabel('τ', fontsize=11, fontweight='bold')
ax.set_title(f'Locations of Top 10% Errors (threshold: {top_10_threshold:.2e})', fontsize=12, fontweight='bold')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('error_localization.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ Visualization saved: error_localization.png')

## 8. SUMMARY REPORT

In [ ]:
print("\n" + "="*80)
print("FINAL EVALUATION REPORT")
print("="*80)

print(f"\n📊 MODEL: {model_file}")
print(f"📊 DATA:  {dat_file}")
print(f"📊 SAMPLES: {N_data:,}")

print(f"\n✅ ACCURACY SUMMARY:")
print(f"   MSE:              {mse:.6e}")
print(f"   RMSE:             {rmse:.6e}")
print(f"   MAE:              {mae:.6e}")
print(f"   Max Abs Error:    {max_abs_error:.6e}")
print(f"   R² Score:         {r2_score:.6f}")

print(f"\n📈 RELATIVE ACCURACY:")
print(f"   Mean Relative Error:  {rel_error_mean:.6e} ({rel_error_mean*100:.4f}%)")
print(f"   Max Relative Error:   {rel_error_max:.6e} ({rel_error_max*100:.4f}%)")
print(f"   Median Abs Error:     {np.median(abs_error_cpu):.6e}")
print(f"   90th Percentile:      {np.percentile(abs_error_cpu, 90):.6e}")

print(f"\n🎯 INTERPRETATION:")
if r2_score >= 0.99:
    print(f"   ✓ Excellent fit (R² ≥ 0.99)")
elif r2_score >= 0.95:
    print(f"   ✓ Very good fit (R² ≥ 0.95)")
elif r2_score >= 0.90:
    print(f"   ✓ Good fit (R² ≥ 0.90)")
else:
    print(f"   ⚠ Needs improvement (R² < 0.90)")

if rel_error_mean < 1e-3:
    print(f"   ✓ Excellent relative accuracy")
elif rel_error_mean < 1e-2:
    print(f"   ✓ Very good relative accuracy")
else:
    print(f"   ⚠ Relative accuracy needs improvement")

print(f"\n✓ Evaluation complete!")
print(f"  - model_evaluation.png")
print(f"  - error_localization.png")